# DS-1000 LoRA + GRPO on AWS Trainium (Neuron)

Train a small **Qwen2.5-Coder** model to solve **DS-1000** data-science programming
problems (Pandas / NumPy / Matplotlib / Sklearn / SciPy / PyTorch / TensorFlow) with
**GRPO** reinforcement learning, on top of a **LoRA** adapter, running on **AWS Trainium**
via the Neuron stack.

> **Why TRL, not SkyRL:** SkyRL has **no Trainium/Neuron backend** — it is CUDA/GPU-only
> (FSDP + vLLM + Ray). The only documented RL path on Trainium is **HuggingFace TRL**, which
> runs on `torch-neuronx` because it rolls out with Transformers `model.generate` (no vLLM).
> LoRA only shrinks the trainable delta; the Neuron requirements are unchanged.

**Reward** = pass/fail on each problem's hidden test harness (`test_execution` + `test_string`),
executed in a sandboxed subprocess. This is *direct* RL (RLVR) — no learned reward model.

**How to use:** run top-to-bottom for a fast **smoke** test (Pandas subset, 1 epoch, G=4).
Set `FULL_RUN=1` before launching the kernel to run all 1,000 problems / 3 epochs / G=8.

**You'll get:** pass@1 **before vs after** RL on a held-out split + the training reward curve.

### 1. Install / verify the RL stack

In [ ]:
import sys
!{sys.executable} -m pip install -q -U transformers trl datasets peft accelerate

import torch, transformers, trl, datasets, peft
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('trl', trl.__version__)
print('datasets', datasets.__version__)
print('peft', peft.__version__)

### 2. Confirm the Neuron device

In [ ]:
import os
os.environ.setdefault('HF_DEACTIVATE_ASYNC_LOAD', '1')   # required on Trainium for the static-cache path

import torch
print('torch', torch.__version__)

neuron_ok = True
for mod in ['torch_neuronx', 'torch_xla']:
    try:
        __import__(mod)
        print(mod, ': available')
    except Exception as e:
        neuron_ok = False
        print(mod, ': NOT available ->', e)

print('CUDA available (should be False on Trainium):', torch.cuda.is_available())
if not neuron_ok:
    print()
    print('WARNING: no Neuron backend detected. This notebook is meant to run on an AWS Trainium')
    print('instance with the AWS DLAMI (torch-neuronx preinstalled). Launch a trn1.2xlarge+ DLAMI')
    print('instance and re-run there.')

### 3. DS-1000 execution sandbox + reward (self-contained)

In [ ]:
# ---- DS-1000 execution sandbox + reward (self-contained; no repo clone needed) ----
# Runs each generated program as a *fresh subprocess* and scores pass/fail by exit code,
# exactly like DS-1000's own evaluator (which treats "no exception" as passed).
#
# WARNING: this is a *timeout + process* guard, NOT a security sandbox. Untrusted model
# code should run in an isolated container/instance, never on your laptop.

import os
import subprocess
import tempfile


def check_correctness(program, timeout, completion_id=None):
    """Run `program` in a fresh subprocess; return {'passed': bool, 'result': str}."""
    fd, path = tempfile.mkstemp(suffix='.py', text=True)
    with os.fdopen(fd, 'w') as f:
        f.write(program)
    try:
        proc = subprocess.run(['python3', path], capture_output=True, text=True,
                              timeout=timeout)
        passed = proc.returncode == 0
        if passed:
            result = 'passed'
        else:
            lines = [l for l in proc.stderr.strip().splitlines() if l.strip()]
            result = lines[-1] if lines else 'failed'
    except subprocess.TimeoutExpired:
        passed, result = False, 'timed out'
    finally:
        try:
            os.unlink(path)
        except OSError:
            pass
    return dict(passed=passed, result=result, completion_id=completion_id)


def postprocess(code):
    """Strip markdown fences / tags the model may emit (mirrors DS-1000's postprocess)."""
    if isinstance(code, list):
        code = code[0]
    code = code.split('</code>')[0]
    code = code.replace('```python', '')
    code = code.split('```')[0]
    code = code.split('\nEND SOLUTION')[0]
    code = code.replace('<code>', '')
    return code.strip()


REWARD_TIMEOUT = 30.0   # seconds per completion during training (short, keeps the loop moving)


def make_test_program(code_context, code):
    return (
        code_context + '\n'
        + f'code = {code!r}\n'
        + 'test_execution(code)\n'
        + ('test_string(code)\n' if 'test_string(' in code_context else '\n')
    )


def ds1000_reward(completions, code_context, **kwargs):
    """TRL reward: 1.0 if the code passes its hidden tests, -0.5 if empty, else 0.0."""
    scores = []
    for completion, ctx in zip(completions, code_context):
        code = postprocess(completion)
        if not code:
            scores.append(-0.5)          # empty / no code -> penalize
            continue
        res = check_correctness(make_test_program(ctx, code), timeout=REWARD_TIMEOUT, completion_id=0)
        scores.append(1.0 if res['passed'] else 0.0)
    return scores

### 4. Experiment config — flip `FULL_RUN` here

In [ ]:
# =====================================================================
#  EXPERIMENT CONFIG
#  Default: SMOKE run (fast). For the FULL benchmark leg:
#      export FULL_RUN=1
#  (or: papermill ds1000_lora_grpo.ipynb out.ipynb -p FULL_RUN 1)
# =====================================================================
import os
FULL_RUN = os.environ.get('FULL_RUN', '0').strip().lower() in ('1', 'true', 'yes', 'on')

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'   # code-specialized, "relatively small"
#  even faster smoke: 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
#  fallback if Coder trips Neuron: 'Qwen/Qwen2.5-1.5B-Instruct'

if FULL_RUN:
    LIBRARY = None              # None = all 1,000 problems across 7 libraries
    MAX_PROBLEMS = None
    NUM_EPOCHS = 3
    GROUP_SIZE = 8              # num_generations (GRPO group)
    PER_DEVICE_BATCH = 2
    MAX_COMPLETION = 1024
    EVAL_SPLIT = 0.2
    REPORT_TO = []              # set to ['wandb'] + export WANDB_API_KEY if desired
else:
    LIBRARY = 'Pandas'          # one library -> fast, self-contained smoke
    MAX_PROBLEMS = 160          # cap the number of problems
    NUM_EPOCHS = 1
    GROUP_SIZE = 4
    PER_DEVICE_BATCH = 2
    MAX_COMPLETION = 512
    EVAL_SPLIT = 0.2
    REPORT_TO = []

print('FULL_RUN =', FULL_RUN)
print('model =', MODEL_ID)
print('library =', LIBRARY or 'all', '| max_problems =', MAX_PROBLEMS or 'all',
      '| epochs =', NUM_EPOCHS, '| group =', GROUP_SIZE)

### 5. Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('tokenizer:', MODEL_ID, '| pad_token =', tokenizer.pad_token)

### 6. Load + prepare the DS-1000 dataset

In [ ]:
from datasets import load_dataset

# DS-1000: 1,000 problems, 7 libraries, single 'test' split on the Hub.
raw = load_dataset('xlangai/DS-1000')['test']
print('total problems:', len(raw))

if LIBRARY is not None:
    raw = raw.filter(lambda r: r['metadata']['library'].lower() == LIBRARY.lower())
    print('library filter', LIBRARY, '->', len(raw))

if MAX_PROBLEMS is not None:
    raw = raw.select(range(min(MAX_PROBLEMS, len(raw))))
    print('capped ->', len(raw))


def make_prompt(row, tok):
    system = ('You are an expert Python programmer. Solve the problem and output ONLY '
              'the Python code (no explanation, no markdown fences).')
    return tok.apply_chat_template(
        [{'role': 'system', 'content': system},
         {'role': 'user', 'content': row['prompt']}],
        tokenize=False, add_generation_prompt=True)


raw = raw.map(lambda r: {
    'prompt': make_prompt(r, tokenizer),
    'library': r['metadata']['library'],
    'problem_id': r['metadata']['problem_id'],
}, remove_columns=['metadata', 'reference_code'])

raw = raw.select_columns(['prompt', 'code_context', 'library', 'problem_id'])

ds = raw.train_test_split(test_size=EVAL_SPLIT, seed=0)
train_ds, eval_ds = ds['train'], ds['test']
print('train', len(train_ds), ' eval', len(eval_ds))

### 7. Evaluation helper (greedy pass@1)

In [ ]:
from collections import defaultdict

EVAL_TIMEOUT = 120.0   # generous timeout for the final before/after measurement


def greedy_eval(model, tok, ds):
    """Greedy-decode every held-out problem and score pass@1 (overall + per library)."""
    passed_total = 0
    per_lib = defaultdict(lambda: [0, 0])
    for i in range(len(ds)):
        enc = tok(ds[i]['prompt'], return_tensors='pt')
        try:
            enc = {k: v.to(model.device) for k, v in enc.items()}
        except Exception:
            pass
        out = model.generate(**enc, max_new_tokens=MAX_COMPLETION, do_sample=False,
                             pad_token_id=tok.eos_token_id)
        comp = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        code = postprocess(comp)
        res = check_correctness(make_test_program(ds[i]['code_context'], code),
                                timeout=EVAL_TIMEOUT, completion_id=i)
        passed_total += int(res['passed'])
        per_lib[ds[i]['library']][0] += int(res['passed'])
        per_lib[ds[i]['library']][1] += 1
    overall = passed_total / len(ds)
    by_lib = {lib: p / t for lib, (p, t) in sorted(per_lib.items())}
    return overall, by_lib

### 8. LoRA + GRPO trainer + BEFORE-RL baseline

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],   # Qwen2.5 attention + MLP
)

training_args = GRPOConfig(
    output_dir='./ds1000-grpo-full' if FULL_RUN else './ds1000-grpo-demo',
    # --- scale (driven by FULL_RUN in section 4) ---
    per_device_train_batch_size=PER_DEVICE_BATCH,
    num_generations=GROUP_SIZE,
    max_completion_length=MAX_COMPLETION,
    max_prompt_length=2048,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=1e-4,          # LoRA lr (higher than the 1e-5 full-model lr)
    beta=0.04,                   # KL coefficient
    temperature=0.7,             # rollout sampling temperature
    # --- Neuron-specific (REQUIRED on Trainium) ---
    cache_implementation='static',                    # fixed decode shapes -> no per-token recompile
    model_init_kwargs={'torch_dtype': 'bfloat16'},   # static cache needs a bf16 model
    pad_to_multiple_of=128,                           # limit the number of compiled shapes
    # --- logging / saving ---
    logging_steps=5,
    report_to=REPORT_TO,
    save_strategy='no',
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=ds1000_reward,
    args=training_args,
    train_dataset=train_ds,
    peft_config=lora_config,        # LoRA on top of GRPO
)

# BEFORE RL: greedy pass@1 (LoRA is freshly init -> identical to the base model).
before_model = trainer.accelerator.unwrap_model(trainer.model)
before_p1, _ = greedy_eval(before_model, tokenizer, eval_ds)
print('BEFORE  pass@1 = %.3f' % before_p1)

### 9. Train

In [ ]:
# first step triggers Neuron compilation for the train shapes -> slow; later steps fast
trainer.train()

### 10. AFTER RL — greedy pass@1 on the same held-out split

In [ ]:
after_model = trainer.accelerator.unwrap_model(trainer.model)
after_p1, after_by_lib = greedy_eval(after_model, tokenizer, eval_ds)
print('AFTER   pass@1 = %.3f' % after_p1)
if len(after_by_lib) > 1:
    for lib, v in after_by_lib.items():
        print('  %-12s %.3f' % (lib, v))

### 11. Results: before vs after + training curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))

ax[0].bar(['Before RL', 'After RL'], [before_p1, after_p1], color=['#999999', '#2ecc71'])
ax[0].set_ylim(0, 1)
ax[0].set_title('DS-1000 pass@1 (eval)')
for i, v in enumerate([before_p1, after_p1]):
    ax[0].text(i, v + 0.02, '%.1f%%' % (v * 100), ha='center', fontweight='bold')

if len(after_by_lib) > 1:
    libs = list(after_by_lib.keys())
    vals = list(after_by_lib.values())
    ax[1].bar(range(len(libs)), vals, color='#3498db')
    ax[1].set_xticks(range(len(libs)))
    ax[1].set_xticklabels(libs, rotation=45, ha='right')
    ax[1].set_ylim(0, 1)
    ax[1].set_title('After-RL pass@1 by library')
else:
    ax[1].set_axis_off()

hist = trainer.state.log_history
steps = [h['step'] for h in hist if 'reward' in h]
rewards = [h['reward'] for h in hist if 'reward' in h]
rstd = [h.get('reward_std', 0.0) for h in hist if 'reward' in h]
if steps:
    ax[2].plot(steps, rewards, marker='o', color='#e74c3c', label='mean reward')
    ax[2].fill_between(steps, np.array(rewards) - np.array(rstd),
                       np.array(rewards) + np.array(rstd), alpha=0.2, color='#e74c3c')
ax[2].set_title('GRPO Training Reward')
ax[2].set_xlabel('step')
ax[2].set_ylabel('reward')
ax[2].legend()

plt.tight_layout()
plt.show()
print('pass@1  %.1f%% -> %.1f%%   (%+.1f pts)' % (before_p1 * 100, after_p1 * 100, (after_p1 - before_p1) * 100))

### Notes / scaling

* **Smoke vs full:** smoke = Pandas subset (160 problems) / 1 epoch / G=4 / no checkpoint;
  full = all 1,000 problems / 3 epochs / G=8 (mirrors the verl reference: beta 0.04, temp 0.7 —
  but LoRA uses lr 1e-4 rather than the full-model 1e-5).
* **Model:** Qwen2.5-Coder-1.5B-Instruct is the code-specialized "relatively small" pick; drop to
  0.5B for a faster smoke, or Qwen2.5-1.5B-Instruct if Coder trips Neuron.
* **Neuron compile:** the first training step compiles the train graph (~1–12 min). `static` cache
  + `pad_to_multiple_of=128` keeps the number of compiled shapes small.
* **Reward cost:** every completion is scored by spawning a subprocess. For the full leg
  (1,000 problems x 8 rollouts x 3 epochs) this is the dominant wall-clock cost; keep
  `REWARD_TIMEOUT` tight (30 s) and consider a `ProcessPoolExecutor` if it is too slow.
* **Safety:** `check_correctness` is a timeout/process guard, NOT a security sandbox. DS-1000
  code legitimately touches the filesystem (Matplotlib/TF/sklearn), so run on an isolated
  Trainium instance and never on a shared machine.
* **LoRA target modules** are Qwen2.5-specific (`q/k/v/o/gate/up/down`); adjust for other archs.
* **Saving:** `save_strategy='no'` (LoRA adapters are a few MB — flip to `'steps'` for checkpoints).